# Phase 1: Exploratory Data Analysis (EDA) on Open Food Facts

**Objective:** Query `data/raw/food.parquet` directly in place using DuckDB without loading 7.9 GB into RAM. Profile data quality, understand nested schemas, analyze country distributions, and establish the subset filtering rule.

In [2]:
import io
import sys

import duckdb

# Ensure UTF-8 output on Windows
if sys.platform == "win32" and isinstance(sys.stdout, io.TextIOWrapper):
    sys.stdout.reconfigure(encoding="utf-8")

# Connect to in-memory DuckDB
con = duckdb.connect()
PARQUET_PATH = '../data/raw/food.parquet'
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.6


## 1. High-Level Summary & Schema Inspection

In [3]:
# Total record count and baseline completeness
con.sql(f'''
    SELECT 
        count(*) AS total_records,
        count(code) AS has_code,
        count(product_name) AS has_name_column,
        count(ingredients_text) AS has_ingredients_column
    FROM '{PARQUET_PATH}'
''').show()

┌───────────────┬──────────┬─────────────────┬────────────────────────┐
│ total_records │ has_code │ has_name_column │ has_ingredients_column │
│     int64     │  int64   │      int64      │         int64          │
├───────────────┼──────────┼─────────────────┼────────────────────────┤
│       4792125 │  4792125 │         4792125 │                4792125 │
└───────────────┴──────────┴─────────────────┴────────────────────────┘



## 2. Inspecting Nested Column Types (`product_name`, `ingredients_text`, `nutriments`)

In [4]:
# Sample nested structure for product name and ingredients
con.sql(f'''
    SELECT 
        code,
        product_name[1].lang AS first_name_lang,
        product_name[1].text AS first_name_text,
        len(ingredients_text) AS num_ingredient_translations,
        len(nutriments) AS num_nutrients_recorded
    FROM '{PARQUET_PATH}'
    WHERE len(product_name) > 0 AND len(ingredients_text) > 0
    LIMIT 5
''').show()

┌───────────────┬─────────────────┬───────────────────────────────────────────────────┬─────────────────────────────┬────────────────────────┐
│     code      │ first_name_lang │                  first_name_text                  │ num_ingredient_translations │ num_nutrients_recorded │
│    varchar    │     varchar     │                      varchar                      │            int64            │         int64          │
├───────────────┼─────────────────┼───────────────────────────────────────────────────┼─────────────────────────────┼────────────────────────┤
│ 0000101209159 │ main            │ Véritable pâte à tartiner noisettes chocolat noir │                           3 │                     14 │
│ 0000105000011 │ main            │ Chamomile Herbal Tea                              │                           3 │                     11 │
│ 0000105000042 │ main            │ Lagg's, herbal tea, peppermint                    │                           3 │                     11 │

## 3. Geographic Distribution: Top 10 Countries

In [5]:
# Top 10 countries represented in dataset
con.sql(f'''
    SELECT 
        list_extract(countries_tags, 1) AS country,
        count(*) AS product_count,
        round(count(*) * 100.0 / sum(count(*)) over (), 2) AS pct_share
    FROM '{PARQUET_PATH}'
    WHERE len(countries_tags) > 0
    GROUP BY country
    ORDER BY product_count DESC
    LIMIT 10
''').show()

┌───────────────────┬───────────────┬───────────┐
│      country      │ product_count │ pct_share │
│      varchar      │     int64     │  double   │
├───────────────────┼───────────────┼───────────┤
│ en:france         │       1239502 │     25.99 │
│ en:united-states  │        963137 │     20.19 │
│ en:germany        │        388907 │      8.15 │
│ en:spain          │        334718 │      7.02 │
│ en:italy          │        251783 │      5.28 │
│ en:united-kingdom │        181000 │       3.8 │
│ en:canada         │        127010 │      2.66 │
│ en:netherlands    │        105505 │      2.21 │
│ en:belgium        │        101430 │      2.13 │
│ en:australia      │         82449 │      1.73 │
└───────────────────┴───────────────┴───────────┘
  10 rows                             3 columns



## 4. Completeness Distribution

In [6]:
# Completeness distribution bins (0.0 to 1.0)
con.sql(f'''
    SELECT 
        floor(completeness * 10) / 10 AS completeness_bin,
        count(*) AS count,
        round(count(*) * 100.0 / (SELECT count(*) FROM '{PARQUET_PATH}'), 2) AS pct
    FROM '{PARQUET_PATH}'
    GROUP BY completeness_bin
    ORDER BY completeness_bin DESC
''').show()

┌──────────────────┬─────────┬────────┐
│ completeness_bin │  count  │  pct   │
│      float       │  int64  │ double │
├──────────────────┼─────────┼────────┤
│              1.1 │    3605 │   0.08 │
│              1.0 │   24691 │   0.52 │
│              0.9 │   70150 │   1.46 │
│              0.8 │  139197 │    2.9 │
│              0.7 │  207623 │   4.33 │
│              0.6 │  366122 │   7.64 │
│              0.5 │  467496 │   9.76 │
│              0.4 │  713505 │  14.89 │
│              0.3 │ 1432298 │  29.89 │
│              0.2 │  860589 │  17.96 │
│              0.1 │  350172 │   7.31 │
│              0.0 │  156661 │   3.27 │
│             NULL │      16 │    0.0 │
└──────────────────┴─────────┴────────┘
  13 rows                   3 columns



## 5. Subset Filtering Funnel Analysis

Measuring the drop at each quality gate to isolate our target ~200k high-quality products.

In [7]:
# Filter funnel evaluation
con.sql(f'''
    SELECT 
        count(*) AS raw_total,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') THEN 1 END) AS valid_barcodes,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 THEN 1 END) AS has_name,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 THEN 1 END) AS has_ingredients,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 AND len(images) > 0 THEN 1 END) AS has_images,
        count(CASE WHEN regexp_matches(code, '^[0-9]{{8}}$|^[0-9]{{12}}$|^[0-9]{{13}}$|^[0-9]{{14}}$') AND len(product_name) > 0 AND len(ingredients_text) > 0 AND len(images) > 0 AND len(nutriments) > 0 AND completeness >= 0.5 THEN 1 END) AS high_quality_complete
    FROM '{PARQUET_PATH}'
''').show()

┌───────────┬────────────────┬──────────┬─────────────────┬────────────┬───────────────────────┐
│ raw_total │ valid_barcodes │ has_name │ has_ingredients │ has_images │ high_quality_complete │
│   int64   │     int64      │  int64   │      int64      │   int64    │         int64         │
├───────────┼────────────────┼──────────┼─────────────────┼────────────┼───────────────────────┤
│   4792125 │        4721147 │  4440824 │         1253334 │     964661 │                822425 │
└───────────┴────────────────┴──────────┴─────────────────┴────────────┴───────────────────────┘

